
# Diffusion MRI Simulators: Hands-on Tutorial

Walk through DMRI's simulator API: build acquisition schemes, simulate single-compartment models (Ball/Stick), and mix them into multi-compartment signals.


In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import jax
import jax.numpy as jnp
import numpy as np

from dmri.simulators import Ball, Stick, Zeppelin, Sphere, Cylinder, MultiCompartment
from dmri.simulators.acquisition_scheme import acquisition_scheme

Duplicate key in file '/Users/manug/src/dmri/dmri/utils/pyloric_black.mplstyle', line 65 ('savefig.facecolor    : black       # Ensures black background when saving')



## 1. Build an acquisition scheme

`acquisition_scheme` bundles **b-values** (diffusion weighting) and **b-vectors** (gradient directions). We sample 100 directions on a unit sphere and linearly spaced b-values up to 3000 s/mm².


In [3]:
bvals = jnp.linspace(0, 3000, 100)
bvecs = jax.random.normal(jax.random.key(0), (100, 3))
bvecs = bvecs / jnp.linalg.norm(bvecs, axis=-1, keepdims=True)

acq = acquisition_scheme(bvals, bvecs)
acq

acquisition_scheme(bvals=Array([   0.     ,   30.30303,   60.60606,   90.90909,  121.21212,
        151.51515,  181.81818,  212.12122,  242.42424,  272.72726,
        303.0303 ,  333.33334,  363.63635,  393.9394 ,  424.24243,
        454.54544,  484.84848,  515.1515 ,  545.4545 ,  575.75757,
        606.0606 ,  636.36365,  666.6667 ,  696.96967,  727.2727 ,
        757.57574,  787.8788 ,  818.1818 ,  848.48486,  878.78784,
        909.0909 ,  939.3939 ,  969.69696, 1000.     , 1030.303  ,
       1060.6061 , 1090.909  , 1121.2122 , 1151.5151 , 1181.8181 ,
       1212.1212 , 1242.4242 , 1272.7273 , 1303.0303 , 1333.3334 ,
       1363.6364 , 1393.9393 , 1424.2424 , 1454.5454 , 1484.8485 ,
       1515.1515 , 1545.4546 , 1575.7576 , 1606.0605 , 1636.3636 ,
       1666.6666 , 1696.9697 , 1727.2727 , 1757.5757 , 1787.8788 ,
       1818.1818 , 1848.4849 , 1878.7878 , 1909.091  , 1939.3939 ,
       1969.6969 , 2000.     , 2030.303  , 2060.606  , 2090.9092 ,
       2121.2122 , 2151.5151 , 2181.8


## 2. Single-compartment examples

### Ball: isotropic diffusion
The Ball model represents free water with scalar diffusivity \(\lambda\).


In [4]:
lam = 0.0018  # mm^2/s
ball = Ball(lam)
ball_signal = ball.signal(acq)
ball_signal

Array([1.        , 0.94691545, 0.8966489 , 0.8490507 , 0.8039793 ,
       0.7613004 , 0.7208871 , 0.68261915, 0.64638263, 0.6120697 ,
       0.5795783 , 0.5488116 , 0.51967824, 0.49209136, 0.4659689 ,
       0.4412332 , 0.4178105 , 0.39563125, 0.37462935, 0.35474232,
       0.33591098, 0.3180793 , 0.3011942 , 0.28520548, 0.2700655 ,
       0.25572917, 0.24215391, 0.22929926, 0.21712703, 0.20560095,
       0.19468674, 0.18435186, 0.17456563, 0.1652989 , 0.15652409,
       0.14821507, 0.14034715, 0.13289689, 0.12584211, 0.11916185,
       0.11283618, 0.10684635, 0.10117444, 0.09580366, 0.09071794,
       0.08590223, 0.08134218, 0.07702415, 0.07293536, 0.06906361,
       0.0653974 , 0.06192581, 0.05863851, 0.05552572, 0.05257815,
       0.04978708, 0.04714415, 0.04464152, 0.04227175, 0.04002777,
       0.03790293, 0.03589086, 0.03398561, 0.03218149, 0.03047316,
       0.02885551, 0.02732372, 0.02587326, 0.02449979, 0.02319923,
       0.02196771, 0.02080156, 0.01969732, 0.0186517 , 0.01766


### Stick: orientation dependence
The Stick has zero radius and attenuates based on the projection of the gradient onto its orientation \(oldsymbol{\mu}\).


In [5]:
stick_aligned = Stick(jnp.array([0.0, 0.0]), lam_par=0.0015)
stick_tilted = Stick(jnp.array([jnp.pi/4, 0.0]), lam_par=0.0015)

signal_aligned = stick_aligned.signal(acq)
signal_tilted = stick_tilted.signal(acq)
signal_aligned, signal_tilted

(Array([1.        , 0.9572091 , 0.9578222 , 0.9464754 , 0.8468933 ,
        0.97745407, 0.87614214, 0.9438814 , 0.7102609 , 0.8739209 ,
        0.9999207 , 0.9632941 , 0.99991703, 0.78534544, 0.9984902 ,
        0.5481609 , 0.99186134, 0.93975353, 0.77369714, 0.9157416 ,
        0.4425291 , 0.49684963, 0.7955569 , 0.9398087 , 0.6864079 ,
        0.7190763 , 0.36802602, 0.8704907 , 0.30084246, 0.3865181 ,
        0.50405556, 0.9043103 , 0.36823285, 0.430203  , 0.4992013 ,
        0.41188395, 0.9505407 , 0.4148516 , 0.32775715, 0.30409515,
        0.3600371 , 0.93847394, 0.16291822, 0.49232244, 0.3164528 ,
        0.90784794, 0.15787415, 0.18246663, 0.6416904 , 0.581278  ,
        0.41219795, 0.25913328, 0.14548665, 0.823272  , 0.43847305,
        0.26121762, 0.21678412, 0.798972  , 0.14585611, 0.9878126 ,
        0.99653244, 0.24596141, 0.3641718 , 0.08161873, 0.98850083,
        0.99731475, 0.96942925, 0.53982806, 0.90829074, 0.7479365 ,
        0.9683095 , 0.2612221 , 0.29651603, 0.03


## 3. Multi-compartment mixture

Combine compartments by subclassing `MultiCompartment`. Here we mix Ball + Stick with equal fractions to illustrate how fractions change the decay.

In addition to the compartments that emulate the signal we can additionally add `NoiseCompoartments`


In [6]:
class BallStick(MultiCompartment):
    model_types = [Ball, Stick]
    noise_types = []
    fraction_prior = jnp.ones(2)

theta = jax.random.normal(jax.random.key(123), (BallStick.theta_dim,))
model = BallStick.from_theta(theta)
signal_mix = model.signal(acq)
signal_mix

Array([1.0000000e+00, 7.9079759e-01, 6.3518363e-01, 4.9241325e-01,
       4.0130746e-01, 3.3247811e-01, 2.7260694e-01, 2.2117570e-01,
       1.7294408e-01, 1.5801731e-01, 1.3371107e-01, 8.3483547e-02,
       8.5453808e-02, 9.0682030e-02, 5.3502202e-02, 5.2849419e-02,
       6.0869686e-02, 2.2037577e-02, 1.7311087e-02, 5.6961257e-02,
       3.4549102e-02, 1.5063046e-02, 1.0068651e-02, 3.0398848e-02,
       5.0531574e-02, 1.4373950e-02, 3.3913339e-03, 5.2280616e-02,
       1.1627597e-02, 3.1419732e-02, 2.2833554e-02, 6.7291413e-03,
       8.6688844e-04, 2.8902646e-02, 5.0306652e-02, 4.9645592e-02,
       8.4314216e-03, 5.0503936e-02, 1.9294850e-03, 5.1313639e-04,
       3.6439754e-02, 1.8406693e-02, 6.2490662e-04, 5.0620541e-02,
       4.8290472e-02, 7.3758303e-03, 4.6940767e-03, 8.6877262e-04,
       2.4984682e-02, 5.0707784e-02, 4.2315304e-02, 2.9820157e-02,
       2.1029970e-04, 5.0721120e-02, 3.9919402e-02, 3.0232099e-05,
       2.2619931e-04, 4.9481042e-02, 2.6016902e-02, 1.6442420e


## 4. Next steps
- Swap in `Sphere` or `Cylinder` to explore restricted diffusion.
- Add noise compartments (e.g., `BoundedGaussianNoise`) when simulating magnitude data.
- Use `model_mask` in `MultiCompartment.from_theta` to toggle components without changing the parameter layout.

For a deeper API tour, see `docs/guides/simulators.md`.
